# **Calibración de la cámara** 📷⚙️ — Blackjack Computer Vision

Mismo procedimiento que en la **Sesión 2 del laboratorio** (calibración con tablero de ajedrez,
método de Zhang), aplicado a la cámara del proyecto. El resultado es el modelo de cámara (`K`,
coeficientes de distorsión y error RMS) que usa el resto del pipeline para corregir la distorsión
antes de detectar las cartas y calcular sus homografías.

**Importante.** `K` solo vale para la cámara y la **resolución** con las que se hicieron las fotos.
Si después se graba vídeo con otra resolución o con otro recorte (el modo vídeo del móvil suele
recortar el sensor), hay que calibrar con fotogramas de ese vídeo (`scripts/calibrate.py --source
video.mp4`). Las gafas se calibrarán con fotogramas del propio stream (ver `docs/calibration.md`).

## 0. Preparación

In [ ]:
import os
import sys
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Locate the repository root and make the project package importable.
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from blackjack_vision.calibration.camera_model import CameraModel
from blackjack_vision.calibration.chessboard import (
    ChessboardSpec,
    calibrate,
    subpixel_half_window,
)
from blackjack_vision.calibration.undistort import Undistorter
from blackjack_vision.capture.sources import IMAGE_EXTENSIONS, read_image, write_image

print("OpenCV", cv2.__version__)

### Parámetros

Revisa esta celda antes de ejecutar:

* `PHOTOS_DIR`: carpeta con las fotos del tablero (git la ignora).
* `CAMERA_NAME`: nombre del modelo; se guardará en `data/calibration/<CAMERA_NAME>.json`.
* `PATTERN`: esquinas **interiores** del tablero `(columnas, filas)`. Un tablero de 10 × 7
  cuadrados tiene 9 × 6 esquinas interiores. Con `None` se detecta automáticamente.
* `SQUARE_SIZE_MM`: lado de un cuadrado medido con una regla. Solo escala los extrínsecos; `K` no
  depende de él.

In [ ]:
PHOTOS_DIR = Path(os.environ.get("CALIB_PHOTOS", ROOT / "data" / "calibration" / "fotos"))
CAMERA_NAME = os.environ.get("CALIB_NAME", "movil")
PATTERN = None  # e.g. (9, 6); None = detect automatically
SQUARE_SIZE_MM = float(os.environ.get("CALIB_SQUARE_MM", "25.0"))

if os.environ.get("CALIB_PATTERN"):
    PATTERN = tuple(int(v) for v in os.environ["CALIB_PATTERN"].lower().split("x"))

OUTPUT_DIR = ROOT / "outputs" / "calibration" / CAMERA_NAME  # figures for the report
MODEL_PATH = ROOT / "data" / "calibration" / f"{CAMERA_NAME}.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Photos:", PHOTOS_DIR)
print("Output:", OUTPUT_DIR)

Funciones de apoyo para visualizar. Como vimos en la **Pregunta B.2 del Lab 1**, OpenCV guarda
las imágenes en **BGR** y matplotlib espera **RGB**, así que hay que convertirlas antes de
mostrarlas (si no, los rojos salen azules). En el notebook usamos matplotlib en lugar de
`cv2.imshow` para que las figuras queden dentro del notebook y se puedan exportar a la memoria.

In [ ]:
def to_rgb(image: np.ndarray) -> np.ndarray:
    return image if image.ndim == 2 else cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def show_grid(images, titles=None, columns=5, size=3.2, save_as=None):
    """Show several BGR images in a grid (and optionally save the figure)."""
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * size, rows * size * 0.8))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for i, (ax, image) in enumerate(zip(np.atleast_1d(axes).ravel(), images)):
        ax.imshow(to_rgb(image), cmap="gray")
        if titles:
            ax.set_title(titles[i], fontsize=9)
    fig.tight_layout()
    if save_as:
        fig.savefig(OUTPUT_DIR / save_as, dpi=150)
    plt.show()


def natural_key(path: Path):
    import re

    return [int(p) if p.isdigit() else p.lower() for p in re.split(r"(\d+)", path.name)]

## **Apartado A: Calibración de la cámara**

### **Tarea A.1:** Cargar las imágenes ``load_images()``

Usamos `read_image` del proyecto en lugar de `cv2.imread`: `cv2.imread` no abre rutas con
caracteres no ASCII en Windows (por ejemplo `C:/Users/José/...`).

Todas las fotos tienen que tener la **misma resolución y orientación**. Si el móvil guardó
algunas en vertical y otras en horizontal, nos quedamos con la orientación mayoritaria: girar
las demás 90° cambiaría el significado de `cx` y `cy`.

In [ ]:
def load_images(filenames):
    return [read_image(filename) for filename in filenames]


imgs_path = sorted(
    (p for p in PHOTOS_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS), key=natural_key
)
imgs = load_images(imgs_path)
names = [p.name for p in imgs_path]

unreadable = [n for n, im in zip(names, imgs) if im is None]
pairs = [(n, im) for n, im in zip(names, imgs) if im is not None]
sizes = Counter((im.shape[1], im.shape[0]) for _, im in pairs)
IMAGE_SIZE = sizes.most_common(1)[0][0]  # (width, height)
discarded = [n for n, im in pairs if (im.shape[1], im.shape[0]) != IMAGE_SIZE]
names = [n for n, im in pairs if (im.shape[1], im.shape[0]) == IMAGE_SIZE]
imgs = [im for _, im in pairs if (im.shape[1], im.shape[0]) == IMAGE_SIZE]

print(f"{len(imgs_path)} files, {len(imgs)} images of {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} px")
print("Sizes found:", dict(sizes))
if unreadable:
    print("Unreadable:", unreadable)
if discarded:
    print("Discarded (different size/orientation):", discarded)
show_grid(imgs, names, save_as="00_photos.png")

### **Tarea A.2:** Detectar las esquinas con ``cv2.findChessboardCorners()`` y refinarlas con ``cv2.cornerSubPix()``

Las fotos del móvil son grandes (12 Mpx o más) y `findChessboardCorners` es lento a esa
resolución. Por eso buscamos el patrón en una copia reducida (lado mayor de 1600 px) y después
refinamos las esquinas con `cornerSubPix` **sobre la imagen original**, de modo que la precisión
final es la de resolución completa.

La ventana de `cornerSubPix` se adapta al tamaño aparente de los cuadrados
(`subpixel_half_window`): una ventana fija de 11 × 11 en una imagen pequeña podría abarcar dos
esquinas.

In [ ]:
DETECTION_MAX_SIDE = 1600
FLAGS = cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]
scale = min(1.0, DETECTION_MAX_SIDE / max(IMAGE_SIZE))
imgs_small = [
    cv2.resize(g, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else g
    for g in imgs_gray
]
print(f"Detection scale: {scale:.3f}")


def detect_pattern(small_images, max_tries=4):
    """Find the inner-corner count by trying the larger boards first."""
    candidates = sorted(
        ((c, r) for c in range(3, 14) for r in range(3, 11) if c >= r),
        key=lambda p: p[0] * p[1],
        reverse=True,
    )
    for pattern in candidates:
        hits = sum(
            cv2.findChessboardCorners(g, pattern, flags=FLAGS | cv2.CALIB_CB_FAST_CHECK)[0]
            for g in small_images[:max_tries]
        )
        if hits >= min(2, len(small_images)):
            return pattern
    raise RuntimeError("No chessboard found: set PATTERN by hand")


if PATTERN is None:
    PATTERN = detect_pattern(imgs_small)
    print("Detected pattern (inner corners):", PATTERN)
spec = ChessboardSpec(PATTERN, SQUARE_SIZE_MM)

In [ ]:
corners = []  # (found, initial corners in full-resolution pixels), like in the lab
for g in imgs_small:
    found, c = cv2.findChessboardCorners(g, PATTERN, flags=FLAGS)
    corners.append((found, c / scale if found else None))

corners_refined = []
for gray, (found, c) in zip(imgs_gray, corners):
    if not found:
        corners_refined.append(None)
        continue
    half = subpixel_half_window(c, spec)
    corners_refined.append(cv2.cornerSubPix(gray, c.copy(), (half, half), (-1, -1), criteria))

for name, (found, _) in zip(names, corners):
    print(f"{name:>24}: {'OK' if found else 'pattern NOT found'}")
print(f"\nPattern found in {sum(f for f, _ in corners)} / {len(imgs)} images")

### **Tarea A.3:** Comprobar las detecciones con ``cv2.drawChessboardCorners()``

La primera esquina se dibuja en rojo y las filas siguen los colores del arcoíris. Hay que
comprobar que **el orden es el mismo en todas las fotos**; si en alguna la primera esquina
aparece en el extremo opuesto, el tablero es simétrico (mismo número par/impar de cuadrados en
ambos lados) y `calibrateCamera` sigue funcionando, pero los extrínsecos de esa vista saldrán
girados 180°.

In [ ]:
imgs_corners = []
for img, (found, _), refined in zip(imgs, corners, corners_refined):
    canvas = img.copy()
    if found:
        cv2.drawChessboardCorners(canvas, PATTERN, refined, True)
    imgs_corners.append(canvas)

show_grid(imgs_corners, names, save_as="01_corners.png")
for i, canvas in enumerate(imgs_corners):
    write_image(OUTPUT_DIR / "corners" / f"{i:02d}_{Path(names[i]).stem}.jpg", canvas)

### **Tarea A.4:** Coordenadas 3D de las esquinas ``get_chessboard_points(chessboard_shape, dx, dy)``

El sistema de referencia está anclado al tablero: origen en la primera esquina, `X` a lo largo
de las columnas, `Y` a lo largo de las filas y `Z = 0` en todo el plano. Es la misma función
del lab; comprobamos que coincide con `ChessboardSpec.object_points()` del proyecto.

In [ ]:
def get_chessboard_points(chessboard_shape, dx, dy):
    points = []
    for j in range(chessboard_shape[1]):
        for i in range(chessboard_shape[0]):
            points.append([i * dx, j * dy, 0])
    return np.array(points, dtype=np.float32)


board_points = get_chessboard_points(PATTERN, SQUARE_SIZE_MM, SQUARE_SIZE_MM)
assert np.allclose(board_points, spec.object_points())

valid = [i for i, (found, _) in enumerate(corners) if found]
valid_names = [names[i] for i in valid]
valid_corners = [corners_refined[i].astype(np.float32) for i in valid]
chessboard_points = [board_points for _ in valid]
print(f"{len(valid)} valid views x {len(board_points)} corners")

### **Tarea A.5:** Calibrar con ``cv2.calibrateCamera()``

Primero exactamente como en el lab. Después con `calibrate()` del proyecto, que llama a
`cv2.calibrateCameraExtended` para obtener además **la incertidumbre de cada parámetro** y el
**error de cada vista**, y descarta automáticamente las vistas con un error anómalo (desenfoque,
detección errónea) antes de recalibrar.

Referencia del curso para el error RMS: **< 0.5 px aceptable, 0.1–0.3 px bueno**. Un RMS bajo
no basta: con pocas poses o poses parecidas el modelo puede sobreajustar.

In [ ]:
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    chessboard_points, valid_corners, IMAGE_SIZE, None, None, criteria=criteria
)
extrinsics = [np.hstack((cv2.Rodrigues(r)[0], t)) for r, t in zip(rvecs, tvecs)]

np.set_printoptions(precision=4, suppress=True)
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients (k1, k2, p1, p2, k3):\n", dist_coeffs.ravel())
print("Root mean squared reprojection error:", round(rms, 4), "px")
print("Extrinsics of the first view [R | t] (t in mm):\n", extrinsics[0])

In [ ]:
result = calibrate(valid_corners, spec, IMAGE_SIZE, view_names=valid_names, name=CAMERA_NAME)
camera = result.camera
print(result.report())

#### Comprobaciones de calidad

1. **Error por vista**: ninguna vista debería destacar mucho sobre las demás.
2. **Cobertura**: las esquinas detectadas deben cubrir toda la imagen, sobre todo **bordes y
   esquinas**, que es donde la distorsión es mayor. Una zona vacía es una zona donde el modelo
   extrapola.
3. **Variedad de poses**: el tablero debe aparecer inclinado en los dos ejes (±30–45°); con
   solo vistas frontales la focal y el punto principal quedan mal determinados.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

order = np.argsort(result.per_view_errors)[::-1]
axes[0].bar(range(len(order)), result.per_view_errors[order], color="tab:blue")
axes[0].axhline(result.rms, color="tab:red", ls="--", label=f"RMS {result.rms:.3f} px")
axes[0].set_xticks(
    range(len(order)), [result.view_names[i] for i in order], rotation=90, fontsize=7
)
axes[0].set_ylabel("RMS reprojection error [px]")
axes[0].set_title("Error per view")
axes[0].legend()

all_points = np.concatenate(
    [valid_corners[valid_names.index(n)].reshape(-1, 2) for n in result.view_names]
)
axes[1].scatter(all_points[:, 0], all_points[:, 1], s=2, alpha=0.5)
axes[1].set_xlim(0, IMAGE_SIZE[0])
axes[1].set_ylim(IMAGE_SIZE[1], 0)
axes[1].set_aspect("equal")
axes[1].set_title("Coverage of the detected corners")
hist, _, _ = np.histogram2d(
    all_points[:, 0], all_points[:, 1], bins=(4, 3), range=[[0, IMAGE_SIZE[0]], [0, IMAGE_SIZE[1]]]
)
print(f"Image cells (4x3) without any corner: {int((hist == 0).sum())} of 12")

tilts = []
for r in result.rvecs:
    normal = cv2.Rodrigues(r)[0][:, 2]
    tilts.append(np.degrees(np.arccos(abs(normal[2]))))
axes[2].hist(tilts, bins=np.arange(0, 75, 5), color="tab:green")
axes[2].set_xlabel("Board tilt with respect to the image plane [deg]")
axes[2].set_title("Pose variety")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_quality.png", dpi=150)
plt.show()
print(
    f"Tilt: min {min(tilts):.0f} deg, median {np.median(tilts):.0f} deg, max {max(tilts):.0f} deg"
)

### **Pregunta A.2:** Diferencias entre ``cv2.findChessboardCorners()`` y ``cv2.cornerSubPix()``

`findChessboardCorners` localiza las esquinas a nivel de píxel (y aquí, además, en la imagen
reducida). `cornerSubPix` las desplaza hasta el punto donde el gradiente de la imagen es
ortogonal al vector que une cada píxel de la ventana con la esquina, lo que da precisión
subpíxel. Medimos cuánto mueve cada esquina:

In [ ]:
shifts = []
for (found, initial), refined in zip(corners, corners_refined):
    if found:
        shifts.append(np.linalg.norm(refined.reshape(-1, 2) - initial.reshape(-1, 2), axis=1))
for name, s in zip(valid_names, shifts):
    print(f"{name:>24}: mean shift = {s.mean():.3f} px (max {s.max():.3f})")

first = valid[0]
initial = corners[first][1].reshape(-1, 2)
refined = corners_refined[first].reshape(-1, 2)
x, y = np.round(refined[0]).astype(int)
r = max(12, int(0.6 * np.linalg.norm(refined[1] - refined[0])))
crop = imgs[first][max(0, y - r) : y + r, max(0, x - r) : x + r]
zoom = cv2.resize(crop, None, fx=8, fy=8, interpolation=cv2.INTER_NEAREST)
offset = np.array([max(0, x - r), max(0, y - r)])
for point, colour in ((initial[0], (0, 0, 255)), (refined[0], (0, 255, 0))):
    centre = tuple(int(v) for v in np.round((point - offset + 0.5) * 8))
    cv2.circle(zoom, centre, 6, colour, -1)
write_image(OUTPUT_DIR / "03_subpix_zoom.png", zoom)
plt.figure(figsize=(5, 5))
plt.imshow(to_rgb(zoom))
plt.title("First corner: findChessboardCorners (red) vs cornerSubPix (green)")
plt.axis("off")
plt.show()

### **Pregunta A.3:** Número mínimo de imágenes necesarias para calibrar

Calibramos con las primeras `n` vistas (en orden aleatorio) y medimos el error de reproyección
sobre **todas** las vistas: para cada vista se estima su pose con `solvePnP` usando los
intrínsecos obtenidos con `n` vistas. Así se ve a partir de cuántas imágenes el modelo deja de
mejorar (el error sobre las propias vistas de entrenamiento siempre parece bueno, incluso con 3).

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(valid_corners))
num_images, errors = [], []
for n in range(3, len(order) + 1):
    subset = order[:n]
    _, K_n, dist_n, _, _ = cv2.calibrateCamera(
        [chessboard_points[i] for i in subset],
        [valid_corners[i] for i in subset],
        IMAGE_SIZE,
        None,
        None,
        criteria=criteria,
    )
    total = 0.0
    for i in range(len(valid_corners)):
        _, rvec, tvec = cv2.solvePnP(chessboard_points[i], valid_corners[i], K_n, dist_n)
        projected, _ = cv2.projectPoints(chessboard_points[i], rvec, tvec, K_n, dist_n)
        residual = valid_corners[i].reshape(-1, 2) - projected.reshape(-1, 2)
        total += float((residual**2).sum())
    num_images.append(n)
    errors.append(np.sqrt(total / (len(valid_corners) * len(board_points))))

plt.figure(figsize=(8, 4))
plt.bar(num_images, errors, alpha=0.4)
plt.plot(num_images, errors, "r-o")
plt.axhline(0.5, color="gray", ls="--", label="0.5 px (acceptable)")
plt.xlabel("Number of images used to calibrate")
plt.ylabel("RMS error over all views [px]")
plt.title("Calibration error vs number of images")
plt.legend()
plt.savefig(OUTPUT_DIR / "04_error_vs_images.png", dpi=150)
plt.show()

### **Pregunta A.4:** ¿Qué modelo de distorsión permiten estimar estas fotos?

Un RMS bajo sobre las propias fotos no garantiza un buen modelo: con muchos coeficientes y pocas
restricciones `calibrateCamera` puede ajustar el ruido. Para saberlo calibramos con distintos
modelos, de más a menos parámetros, y medimos el error de **validación cruzada dejando una foto
fuera** (se calibra sin ella y se mide el error de reproyección en ella).

Si un modelo con menos parámetros predice igual o mejor, los parámetros extra no están
determinados por los datos. En particular, si el modelo **sin distorsión** iguala al completo,
las fotos no contienen información sobre la distorsión (típicamente porque el tablero nunca
llega a los bordes de la imagen).

In [ ]:
K_guess = np.array(
    [
        [max(IMAGE_SIZE), 0, (IMAGE_SIZE[0] - 1) / 2],
        [0, max(IMAGE_SIZE), (IMAGE_SIZE[1] - 1) / 2],
        [0, 0, 1],
    ]
)
NO_TANGENT_K3 = cv2.CALIB_FIX_K3 | cv2.CALIB_ZERO_TANGENT_DIST
SQUARE_PIXELS = cv2.CALIB_FIX_ASPECT_RATIO
CENTRED = cv2.CALIB_FIX_PRINCIPAL_POINT
models = {
    "full (k1 k2 p1 p2 k3)": 0,
    "k1 k2": NO_TANGENT_K3,
    "k1 k2, fx = fy": NO_TANGENT_K3 | SQUARE_PIXELS,
    "k1, fx = fy": NO_TANGENT_K3 | cv2.CALIB_FIX_K2 | SQUARE_PIXELS,
    "no distortion, fx = fy, centred": NO_TANGENT_K3
    | cv2.CALIB_FIX_K1
    | cv2.CALIB_FIX_K2
    | SQUARE_PIXELS
    | CENTRED,
}


def calibrate_subset(indices, flags):
    return cv2.calibrateCamera(
        [board_points] * len(indices),
        [valid_corners[i] for i in indices],
        IMAGE_SIZE,
        K_guess.copy(),
        np.zeros(5),
        flags=flags | cv2.CALIB_USE_INTRINSIC_GUESS,
    )


comparison = {}
for model_name, flags in models.items():
    rms_m, K_m, dist_m, _, _ = calibrate_subset(range(len(valid_corners)), flags)
    held_out = []
    for i in range(len(valid_corners)):
        _, K_i, dist_i, _, _ = calibrate_subset(
            [j for j in range(len(valid_corners)) if j != i], flags
        )
        _, rvec, tvec = cv2.solvePnP(board_points, valid_corners[i], K_i, dist_i)
        projected, _ = cv2.projectPoints(board_points, rvec, tvec, K_i, dist_i)
        residual = valid_corners[i].reshape(-1, 2) - projected.reshape(-1, 2)
        held_out.append(np.sqrt((residual**2).sum(axis=1).mean()))
    comparison[model_name] = float(np.mean(held_out))
    print(
        f"{model_name:34s} train {rms_m:.3f} px | leave-one-out {np.mean(held_out):.3f} px | "
        f"f=({K_m[0, 0]:.0f}, {K_m[1, 1]:.0f}) c=({K_m[0, 2]:.0f}, {K_m[1, 2]:.0f}) "
        f"dist={np.round(dist_m.ravel()[:5], 3)}"
    )

full = comparison["full (k1 k2 p1 p2 k3)"]
none = comparison["no distortion, fx = fy, centred"]
if none <= 1.05 * full:
    print(
        "\nWARNING: a model without distortion predicts as well as the full one, so these "
        "photos carry no information about the lens distortion. Retake them with the board "
        "near the borders and corners of the image (see the conclusions)."
    )

## **Apartado B: Corrección de la distorsión**

Con `K` y los coeficientes de distorsión se precalculan los mapas de `initUndistortRectifyMap`
una sola vez y cada fotograma se corrige con un único `cv2.remap` (lo que hará el asistente en
tiempo real). `alpha` decide qué hacer con los bordes:

* `alpha = 0`: recorta para que todos los píxeles sean válidos (sin bordes negros).
* `alpha = 1`: conserva todos los píxeles de la imagen original (bordes curvos negros).

Las líneas rectas del mundo (bordes del tablero, de la mesa, de una puerta) deben verse rectas
tras la corrección, también cerca de los bordes de la imagen.

In [ ]:
view_errors = dict(zip(result.view_names, result.per_view_errors))
examples = sorted(
    view_errors,
    key=lambda n: (
        -np.abs(valid_corners[valid_names.index(n)].reshape(-1, 2) - np.array(IMAGE_SIZE) / 2).max()
    ),
)[:2]

panels, titles = [], []
for alpha in (0.0, 1.0):
    undistorter = Undistorter(camera, alpha=alpha)
    for name in examples:
        panels.append(undistorter(imgs[names.index(name)]))
        titles.append(f"{name} - alpha={alpha:.0f}")
originals = [imgs[names.index(n)] for n in examples]
show_grid(
    originals + panels,
    [f"{n} - original" for n in examples] + titles,
    columns=2,
    size=6,
    save_as="05_undistorted.png",
)

Comprobación numérica de la rectitud: las esquinas de cada fila del tablero están sobre una
recta en el mundo, así que después de corregirlas (`undistort_points`) deberían quedar
alineadas. Medimos la distancia media de cada esquina a la recta ajustada a su fila, antes y
después de la corrección.

In [ ]:
undistorter = Undistorter(camera, alpha=0.0)
columns, rows = PATTERN


def straightness(points):
    grid = points.reshape(rows, columns, 2)
    residuals = []
    for line in grid:
        vx, vy, x0, y0 = cv2.fitLine(line.astype(np.float32), cv2.DIST_L2, 0, 0.01, 0.01).ravel()
        normal = np.array([-vy, vx])
        residuals.append(np.abs((line - [x0, y0]) @ normal))
    return float(np.mean(residuals))


before = [straightness(c.reshape(-1, 2)) for c in valid_corners]
after = [straightness(undistorter.undistort_points(c.reshape(-1, 2))) for c in valid_corners]
print(
    f"Mean distance of the corners to the line of their row: "
    f"{np.mean(before):.3f} px before, {np.mean(after):.3f} px after correction"
)

## **Apartado C: Guardar el modelo de cámara**

El modelo se guarda en JSON (legible y pequeño) en `data/calibration/`; esos ficheros sí se
suben a git. Las fotos y las figuras no.

Si el vídeo se graba a otra resolución **con el mismo campo de visión** (un simple reescalado),
`camera.scaled((ancho, alto))` adapta `K`. Si hay recorte, hay que recalibrar con fotogramas de
ese vídeo.

In [ ]:
camera.save(MODEL_PATH)
print("Saved", MODEL_PATH.relative_to(ROOT))
print(camera.summary())

loaded = CameraModel.load(MODEL_PATH)
assert np.allclose(loaded.camera_matrix, camera.camera_matrix)

print(f"\nVerdict: RMS {result.rms:.3f} px -> {result.quality}")
print("Figures for the report in", OUTPUT_DIR.relative_to(ROOT))

## Conclusiones

*(Completar tras ejecutar con las fotos definitivas.)*

* Error RMS final y vistas descartadas.
* ¿Qué zonas de la imagen quedan sin cubrir y cómo afecta?
* ¿A partir de cuántas imágenes se estabiliza el error (Pregunta A.3)?
* ¿Qué modelo de distorsión justifican los datos (Pregunta A.4)?
* ¿Se ven rectos los bordes tras la corrección? ¿Cuánto mejora la rectitud medida?

### Primera sesión de fotos (18/11/2026): por qué hay que repetirla

25 fotos de un tablero A4 de 7 × 5 cuadrados (6 × 4 esquinas interiores), enviadas por WhatsApp
(1536 × 2048 px, recomprimidas y sin EXIF). El tablero ocupa ~10 % de la imagen y siempre está
en el centro: 10 de las 12 zonas de la imagen no tienen ninguna esquina. Resultado: RMS 0,65 px,
`fx` y `fy` distintos un 3,5 %, coeficientes de distorsión sin sentido físico (`k2 = 0,71`,
`k3 = -0,97`) y un modelo **sin distorsión** que predice igual que el completo (validación
cruzada 0,71 frente a 0,73 px). Es decir, esas fotos no contienen información sobre la
distorsión y la focal solo queda acotada a ±5 % (1017–1134 px).

### Cómo repetir las fotos

1. **Calibrar la cámara que se va a usar, en el modo en que se va a usar.** Para el móvil como
   cámara de respaldo lo mejor es grabar un vídeo de 30–60 s en el modo de vídeo definitivo y
   pasarlo por `scripts/calibrate.py --source video.mp4`, que elige los fotogramas.
2. **No pasar las fotos por WhatsApp** (reduce y recomprime): enviarlas como *documento*, por
   Drive o por cable.
3. **Tablero grande en la imagen**: acercarse hasta que ocupe un 40–80 % de la foto.
4. **Llevarlo a los bordes y esquinas**: al menos 2–3 fotos con el tablero en cada esquina y en
   cada borde, además de algunas centradas (unas 25 en total).
5. **Variar la inclinación** (±30–45° en ambos ejes) y la distancia.
6. **Misma lente y zoom en todas** (sin zoom digital; ojo con los móviles que cambian solos a la
   lente gran angular o al modo macro al acercarse) y misma orientación (todas verticales o todas
   horizontales).
7. Tablero **plano y rígido** (pegado a un cartón), mate, completo en la imagen con su margen
   blanco, y fotos nítidas.